In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors

### Brownian motion in $\mathbb{R}^2$

In [ ]:
# sample the SDE using Euler-Maruyama scheme
def sample(dim=2, T=1.0, N=10000, seed=42):
    rng = np.random.default_rng(seed=seed)
    X = [0, 0]
    traj = [X]
    delta_t = T / N
    save = 10
    for i in range(N):
        b = rng.normal(size=(dim,))
        X = X + np.sqrt(delta_t) * b
        if i % save==0:
            traj.append(X)
    return np.array(traj)

trajectory = sample(T=1,seed=128)
plt.plot(trajectory[:,0], trajectory[:,1])
plt.xlim([-2,2])
plt.ylim([-2,2])

### Brownian dynamics 

$$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t$$

with potential

$$V(x) = (x_1^2-1)^2 + 2.0 * (x_1^2+x_2-1)^2$$ 

first, define $V$ and its gradient

In [ ]:
# a potential function
def V(X):
    return (X[0]**2 - 1)**2 + 2.0 * (X[0]**2 + X[1] - 1)**2

# gradient of potential function 
def gradV(X):
    return np.array(( 4.0 * X[0] * (X[0]**2 - 1.0 + 2.0*(X[0]**2 + X[1] - 1)), 4.0 * (X[0]**2 + X[1] - 1)) )

show the potential profile

In [ ]:
x = np.arange(-2.5, 2.5, 0.05)
y = np.arange(-2.5, 2.5, 0.05)
X, Y = np.meshgrid(x, y)

plt.figure(figsize = (6, 4))

contour_levels = [0.0, 1.0, 1.5, 2.0, 3.0, 4.0]

# evaluate potential on mesh
V_on_grid = V([X,Y])

fig = plt.figure(figsize=(7,4))
ax = fig.add_subplot(1, 1, 1)

# plot profile by pcolormesh
im = ax.pcolormesh(X, Y, V_on_grid, cmap='coolwarm',shading='auto', vmin=0, vmax=4)

# show contour lines
contours = ax.contour(X, Y, V_on_grid, contour_levels)
ax.clabel(contours, inline=True, fontsize=13,colors='black')

ax.set_aspect('equal')
ax.set_xlabel(r'$x_1$',fontsize=20)
ax.set_ylabel(r'$x_2$',fontsize=20, rotation=0)
ax.tick_params(axis='both', labelsize=20)

ax.set_xticks([-2.0, -1.0, 0, 1.0, 2.0])
ax.set_yticks([-2.0, -1.0, 0, 1.0, 2.0])
ax.set_xlim([-2.5, 2.5])
ax.set_ylim([-2.5, 2.5])

ax.set_title('V',fontsize=25)

# show colorbar
cbar = fig.colorbar(im, ax=ax, shrink=1.0)
cbar.ax.tick_params(labelsize=15)
plt.show()

### Simulate the SDE 

$$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t$$

Question and task:

1. try the code with beta=1.0, 2.0, 4.0
2. what happen when delta_t is increased to 0.01 and 0.1?
3. increase N to $2\times 10^7$ and see the runtime changes.

In [ ]:
# sample the SDE using Euler-Maruyama scheme
import time

def sample(beta=1.0, N=10000, delta_t=0.001, seed=42):
    rng = np.random.default_rng(seed=seed)
    X = [-1, 0]
    dim = 2 
    traj = [X]
    save = 10
    for i in range(N):
        b = rng.normal(size=(dim,))
        X = X - gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b
        if i % save==0:
            traj.append(X)

    return np.array(traj)

seed = 42

t0 = time.time()
trajectory = sample(beta=1.0, N=200000, delta_t=0.001, seed=seed)
t1 = time.time()     
print ("\n\nshape of trajectory array:", trajectory.shape)
print ("runtime: %.1f sec." % (t1-t0))
# compute values of potential along the trajectory
v_traj = np.array([V(x) for x in trajectory])

### plot 

In [ ]:
fig, ax=plt.subplots(1,2,figsize=(12,4))

ret = ax[0].scatter(trajectory[:,0], trajectory[:,1], c=v_traj)

cbar = plt.colorbar(ret, ax=ax[0], shrink=1.0)
cbar.ax.tick_params(labelsize=15)
ax[0].set_xlabel(r'$x_1$',fontsize=20)
ax[0].set_ylabel(r'$x_2$',fontsize=20, rotation=0)
ax[0].tick_params(axis='both', labelsize=20)
ax[0].set_title('trajectory data',fontsize=15)

ax[1].plot(trajectory[:,0])
ax[1].set_xlabel(r'step',fontsize=20)
ax[1].set_ylabel(r'$x_1$',fontsize=20)

plt.show()

### OU Process

Brownian dynamics:
$$dX_t = -\nabla V(X_t) dt + \sqrt{2\beta^{-1}} dB_t$$

With the choice $V(x) = \frac{\kappa |x|^2}{2}$, we get the OU process:

$$dX_t = -\kappa X_t dt + \sqrt{2\beta^{-1}} dB_t$$

Try:

1. beta = 1.0, 0.5, 0.2
2. kappa = 1.0, 0.1, 10.0
 
and see how the result changes.

In [ ]:
# sample the SDE using Euler-Maruyama scheme
def sample(dim=1, beta=1.0, kappa=1.0, N=10000, seed=42):
    rng = np.random.default_rng(seed=seed)
    X = np.zeros(dim)
    traj = [X]
    delta_t = 0.001
    save = 5
    for i in range(N):
        b = rng.normal(size=(dim,))
        X = X - kappa * X * delta_t + np.sqrt(2 * delta_t/beta) * b
        if i % save==0:
            traj.append(X)

    return np.array(traj)

beta = 1.0
kappa = 1.0

# generate a long trajectory, set dim=2
trajectory = sample(dim=2, beta=beta, kappa=kappa, N=1000000, seed=400)
print ("Number of states:", trajectory.shape[0])

OU process in $\mathbb{R}^2$

In [ ]:

fig = plt.figure(figsize=(7,5))
ax = fig.add_subplot(1, 1, 1)

# compute histogram statistics
h, xedges, yedges = np.histogram2d(trajectory[:,0], trajectory[:,1], bins=[100, 100], range=[[-2.5,2.5],[-2.5,2.5]], density=True)

# get the meshgrid
X, Y = np.meshgrid(xedges, yedges)
# plot the histogram, specify the colormap and log scale
im = ax.pcolormesh(X, Y, h.T, cmap='coolwarm', shading='auto')

# show colorbar
cbar = fig.colorbar(im, ax=ax, shrink=1.0)
cbar.ax.tick_params(labelsize=15)

ax.set_aspect('equal')
ax.set_xlim([-2.5, 2.5])
ax.set_ylim([-2.5, 2.5])
ax.set_xlabel(r'$x_1$',fontsize=20)
ax.set_ylabel(r'$x_2$',fontsize=20, rotation=0)
ax.tick_params(axis='both', labelsize=20)
ax.set_xticks([-2.0, -1.0, 0, 1.0, 2.0])
ax.set_yticks([-2.0, -1.0, 0, 1.0, 2.0])
plt.show()